# Engenharia de Dados: construindo um pipeline completo com Python

**Curso:** Pós-Graduação em Ciência de Dados  
**Disciplina:** Engenharia de Dados (ETL) e Big Data  
**Bloco:** Introdução à Engenharia de Dados  
**Ambiente:** Google Colab

Neste notebook, construiremos um pipeline didático para uma instituição de ensino. O exemplo utiliza dados fictícios e mostra como diferentes fontes podem ser coletadas, avaliadas, transformadas, integradas, armazenadas e consumidas.

> **Observação:** os dados são sintéticos. Nenhuma informação pessoal real é utilizada.

## Objetivos de aprendizagem

Ao final da demonstração, o estudante deverá ser capaz de:

1. Identificar dados estruturados, semiestruturados e não estruturados;
2. Reconhecer as etapas de um pipeline de dados;
3. Avaliar problemas de qualidade, como valores ausentes, duplicidades e inconsistências;
4. Aplicar transformações com Python e Pandas;
5. Integrar dados provenientes de fontes diferentes;
6. Armazenar dados tratados em arquivos e em um banco SQLite;
7. Consumir os dados por meio de indicadores, consultas e gráficos.

### Mapa da demonstração

**Origem → Extração → Qualidade → Transformação → Integração → Armazenamento → Consumo**

## 1. Preparação do ambiente

As bibliotecas utilizadas já estão disponíveis no Google Colab. A semente aleatória torna a geração dos dados reproduzível: toda execução completa produzirá os mesmos valores.

In [ ]:
# Biblioteca para trabalhar com caminhos e diretórios de forma portátil.
from pathlib import Path

# Bibliotecas da própria linguagem Python.
import json
import re
import sqlite3
import unicodedata

# Bibliotecas para dados, cálculos e visualizações.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Função do ambiente Jupyter/Colab para exibir tabelas com melhor formatação.
from IPython.display import display

# Configurações para deixar as tabelas mais legíveis durante a apresentação.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

# Semente fixa: garante que os dados fictícios sejam sempre os mesmos.
SEMENTE = 42
gerador = np.random.default_rng(SEMENTE)

# Organização das camadas do pipeline.
# bronze: dados originais, preservados como chegaram.
# silver: dados limpos, padronizados e integrados.
# gold: dados organizados para análise e consumo.
PASTA_BASE = Path("dados_aula")
PASTA_BRONZE = PASTA_BASE / "bronze"
PASTA_SILVER = PASTA_BASE / "silver"
PASTA_GOLD = PASTA_BASE / "gold"

for pasta in [PASTA_BRONZE, PASTA_SILVER, PASTA_GOLD]:
    pasta.mkdir(parents=True, exist_ok=True)

print("Ambiente preparado com sucesso.")
print(f"Diretório principal: {PASTA_BASE.resolve()}")

## 2. Criação das fontes de dados

Para fins didáticos, criaremos três fontes:

| Fonte | Formato | Classificação didática | Exemplo de conteúdo |
|---|---|---|---|
| Sistema acadêmico | CSV | Estruturado | Matrículas e dados cadastrais |
| API de atendimento | JSON | Semiestruturado | Interações e atributos aninhados |
| Registros de atendimento | TXT | Não estruturado | Observações escritas em linguagem natural |

Os arquivos conterão problemas intencionais de qualidade para que seja possível demonstrar o trabalho da Engenharia de Dados.

In [ ]:
# -----------------------------
# 2.1 Fonte estruturada: CSV
# -----------------------------

nomes = [
    "Ana Souza", "Bruno Lima", "Carla Mendes", "Diego Santos", "Elisa Rocha",
    "Felipe Alves", "Gabriela Silva", "Henrique Costa", "Isabela Nunes", "João Melo",
    "Karina Freitas", "Lucas Barros", "Mariana Oliveira", "Nicolas Araújo", "Paula Ribeiro",
    "Rafael Gomes", "Sabrina Martins", "Tiago Ferreira", "Valéria Castro", "William Lopes",
    "Aline Cavalcanti", "Caio Monteiro", "Débora Almeida", "Eduardo Correia", "Fernanda Sales",
    "Gustavo Pires", "Helena Dantas", "Igor Farias", "Juliana Teixeira", "Leandro Moura"
]

# Variações propositais: maiúsculas, minúsculas, abreviações e acentos inconsistentes.
cidades_sujas = ["recife", "RECIFE", "Recife ", "olinda", "OLINDA", "paulista", "Paulista", "JABOATÃO", None]
cursos_sujos = [
    "Ciência de Dados", "ciencia de dados", "CD",
    "Engenharia de Software", "eng. software", "ES",
    "Inteligência Artificial", "inteligencia artificial", "IA"
]
status_sujos = ["Ativa", "ATIVO", "ativa ", "Pendente", "pendente", "Cancelada", "CANCELADO"]

registros_matriculas = []

for indice, nome in enumerate(nomes, start=1):
    aluno_id = 1000 + indice
    data_base = pd.Timestamp("2026-08-01") + pd.Timedelta(days=int(gerador.integers(0, 50)))
    atualizado_em = data_base + pd.Timedelta(days=int(gerador.integers(0, 7)))

    # Geramos formatos de data diferentes para simular sistemas legados.
    if indice % 3 == 0:
        data_matricula = data_base.strftime("%d/%m/%Y")
    elif indice % 3 == 1:
        data_matricula = data_base.strftime("%Y-%m-%d")
    else:
        data_matricula = data_base.strftime("%d-%m-%Y")

    valor = float(gerador.choice([890, 950, 1100, 1250, 1380]))

    # Alguns valores monetários chegam com a formatação brasileira.
    valor_mensalidade = (
        f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        if indice % 4 == 0
        else valor
    )

    registros_matriculas.append(
        {
            "aluno_id": aluno_id,
            "nome": nome if indice % 6 else f"  {nome.upper()}  ",
            "idade": int(gerador.integers(20, 56)),
            "cidade": gerador.choice(cidades_sujas),
            "curso": gerador.choice(cursos_sujos),
            "status_matricula": gerador.choice(status_sujos),
            "data_matricula": data_matricula,
            "valor_mensalidade": valor_mensalidade,
            "email": f"{nome.lower().replace(' ', '.')}@exemplo.com",
            "atualizado_em": atualizado_em.strftime("%Y-%m-%d %H:%M:%S"),
        }
    )

matriculas_origem = pd.DataFrame(registros_matriculas)

# Inserimos problemas de qualidade intencionais.
matriculas_origem.loc[2, "idade"] = None                  # valor ausente
matriculas_origem.loc[7, "idade"] = 115                   # valor impossível
matriculas_origem.loc[4, "email"] = None                  # não devemos inventar e-mail
matriculas_origem.loc[10, "email"] = "email_invalido"    # formato inválido
matriculas_origem.loc[12, "valor_mensalidade"] = None     # valor ausente
matriculas_origem.loc[18, "data_matricula"] = "31/02/2026"  # data impossível

# Duplicidade exata: a linha foi recebida duas vezes.
matriculas_origem = pd.concat(
    [matriculas_origem, matriculas_origem.iloc[[5]].copy()],
    ignore_index=True,
)

# Duplicidade de chave: o mesmo aluno recebeu uma atualização posterior.
registro_atualizado = matriculas_origem.iloc[[1]].copy()
registro_atualizado["status_matricula"] = "Ativa"
registro_atualizado["cidade"] = "Recife"
registro_atualizado["atualizado_em"] = "2026-09-18 10:00:00"
matriculas_origem = pd.concat([matriculas_origem, registro_atualizado], ignore_index=True)

caminho_csv = PASTA_BRONZE / "matriculas.csv"
matriculas_origem.to_csv(caminho_csv, index=False, encoding="utf-8")

# --------------------------------
# 2.2 Fonte semiestruturada: JSON
# --------------------------------

canais = ["WhatsApp", "whatsapp", "E-mail", "EMAIL", "Telefone", "Portal"]
eventos = ["Solicitou informação", "Enviou documento", "Negociou pagamento", "Atualizou cadastro"]
interacoes_api = []

for interacao_id in range(1, 66):
    aluno_id = int(gerador.choice(range(1001, 1031)))
    data_interacao = pd.Timestamp("2026-08-05") + pd.Timedelta(days=int(gerador.integers(0, 45)))

    interacoes_api.append(
        {
            "interacao_id": interacao_id,
            "aluno_id": aluno_id,
            "canal": gerador.choice(canais),
            "evento": gerador.choice(eventos),
            "data_hora": data_interacao.strftime("%Y-%m-%d %H:%M:%S"),
            "detalhes": {
                "duracao_min": int(gerador.integers(1, 25)),
                "origem": gerador.choice(["site", "campanha", "indicação", "orgânico"]),
                "atendente": gerador.choice(["Equipe A", "Equipe B", "Autoatendimento"]),
            },
        }
    )

# Registro órfão: a interação referencia um aluno inexistente na fonte de matrículas.
interacoes_api.append(
    {
        "interacao_id": 999,
        "aluno_id": 9999,
        "canal": "WhatsApp",
        "evento": "Solicitou informação",
        "data_hora": "2026-09-18 09:30:00",
        "detalhes": {"duracao_min": 8, "origem": "site", "atendente": "Equipe A"},
    }
)

caminho_json = PASTA_BRONZE / "interacoes.json"
with open(caminho_json, "w", encoding="utf-8") as arquivo_json:
    json.dump(interacoes_api, arquivo_json, ensure_ascii=False, indent=2)

# -------------------------------
# 2.3 Fonte não estruturada: TXT
# -------------------------------

observacoes_texto = """
ALUNO_ID=1001; Demonstrou interesse em bolsa e pediu retorno por telefone.
ALUNO_ID=1003; Informou dificuldade para enviar os documentos pelo portal.
ALUNO_ID=1005; Solicitou informações sobre acessibilidade no ambiente virtual.
ALUNO_ID=1008; Pediu negociação financeira antes de confirmar a matrícula.
ALUNO_ID=1012; Elogiou o atendimento e confirmou que enviará os documentos.
ALUNO_ID=1015; Ainda possui dúvida sobre o calendário e o início das aulas.
ALUNO_ID=1021; Solicitou bolsa e perguntou sobre formas de pagamento.
ALUNO_ID=1027; Relatou problema de acesso ao portal e pediu suporte.
""".strip()

caminho_txt = PASTA_BRONZE / "observacoes_atendimento.txt"
caminho_txt.write_text(observacoes_texto, encoding="utf-8")

print("Fontes criadas na camada bronze:")
for arquivo in sorted(PASTA_BRONZE.iterdir()):
    print(f"- {arquivo.name} ({arquivo.stat().st_size} bytes)")

## 3. Extração: leitura das fontes

Nesta etapa, os dados são lidos sem alterações destrutivas. Preservar a origem permite auditoria, reprocessamento e comparação com o dado tratado.

In [ ]:
# Lemos o CSV inicialmente como texto para preservar exatamente o que chegou.
matriculas_brutas = pd.read_csv(caminho_csv, dtype="object")

# O JSON possui um objeto aninhado em "detalhes".
# json_normalize transforma os atributos internos em colunas tabulares.
with open(caminho_json, "r", encoding="utf-8") as arquivo_json:
    interacoes_json = json.load(arquivo_json)

interacoes_brutas = pd.json_normalize(interacoes_json, sep="_")

# O TXT é mantido como texto. A interpretação será feita em uma etapa posterior.
observacoes_brutas = caminho_txt.read_text(encoding="utf-8")

print("Amostra da fonte CSV:")
display(matriculas_brutas.head(5))

print("\nAmostra da fonte JSON normalizada:")
display(interacoes_brutas.head(5))

print("\nConteúdo da fonte textual:")
print(observacoes_brutas)

## 4. Diagnóstico de qualidade

Antes de corrigir qualquer problema, precisamos medi-lo. O perfil abaixo mostra volume, duplicidades, tipos e valores ausentes.

In [ ]:
def diagnosticar_qualidade(nome_fonte, dataframe):
    """Exibe um resumo reutilizável da qualidade de um DataFrame."""

    resumo = pd.DataFrame(
        {
            "metrica": ["Linhas", "Colunas", "Duplicidades exatas", "Células ausentes"],
            "valor": [
                dataframe.shape[0],
                dataframe.shape[1],
                int(dataframe.duplicated().sum()),
                int(dataframe.isna().sum().sum()),
            ],
        }
    )

    ausencias = (
        dataframe.isna()
        .sum()
        .rename("quantidade_ausente")
        .to_frame()
        .assign(percentual_ausente=lambda tabela: tabela["quantidade_ausente"] / len(dataframe) * 100)
        .query("quantidade_ausente > 0")
        .sort_values("quantidade_ausente", ascending=False)
    )

    print(f"DIAGNÓSTICO: {nome_fonte}")
    display(resumo)
    print("Tipos identificados:")
    display(dataframe.dtypes.rename("tipo").to_frame())
    print("Valores ausentes por coluna:")
    display(ausencias.round(2) if not ausencias.empty else pd.DataFrame({"mensagem": ["Nenhuma ausência detectada"]}))


diagnosticar_qualidade("Matrículas", matriculas_brutas)
diagnosticar_qualidade("Interações", interacoes_brutas)

print("Duplicidades pela chave de negócio aluno_id:")
display(
    matriculas_brutas[matriculas_brutas.duplicated(subset=["aluno_id"], keep=False)]
    .sort_values(["aluno_id", "atualizado_em"])
)

### Perguntas para a turma

- Uma coluna sem valores ausentes é necessariamente confiável?
- Uma linha repetida é sempre um erro?
- Qual registro deve ser mantido quando o mesmo aluno aparece mais de uma vez?
- Devemos preencher automaticamente todos os valores ausentes?

Essas decisões dependem de regras de negócio, contexto e rastreabilidade.

## 5. Funções de transformação

As funções abaixo concentram regras reutilizáveis. Esse desenho facilita testes, manutenção e aplicação das mesmas regras em novas cargas.

In [ ]:
def remover_acentos(texto):
    """Remove acentos apenas para comparação e criação de chaves de padronização."""
    if pd.isna(texto):
        return None
    texto_normalizado = unicodedata.normalize("NFKD", str(texto))
    return "".join(caractere for caractere in texto_normalizado if not unicodedata.combining(caractere))


def criar_chave_textual(texto):
    """Cria uma representação comparável: sem espaços extras, sem acento e em minúsculas."""
    if pd.isna(texto):
        return None
    texto_sem_acentos = remover_acentos(texto)
    return " ".join(texto_sem_acentos.strip().lower().split())


def padronizar_nome(texto):
    """Remove espaços duplicados e aplica capitalização adequada para a demonstração."""
    if pd.isna(texto):
        return pd.NA
    return " ".join(str(texto).strip().split()).title()


MAPA_CIDADES = {
    "recife": "Recife",
    "olinda": "Olinda",
    "paulista": "Paulista",
    "jaboatao": "Jaboatão dos Guararapes",
}

MAPA_CURSOS = {
    "ciencia de dados": "Ciência de Dados",
    "cd": "Ciência de Dados",
    "engenharia de software": "Engenharia de Software",
    "eng. software": "Engenharia de Software",
    "es": "Engenharia de Software",
    "inteligencia artificial": "Inteligência Artificial",
    "ia": "Inteligência Artificial",
}

MAPA_STATUS = {
    "ativa": "Ativa",
    "ativo": "Ativa",
    "pendente": "Pendente",
    "cancelada": "Cancelada",
    "cancelado": "Cancelada",
}


def padronizar_com_mapa(valor, mapa, padrao="Não informado"):
    """Padroniza categorias por um dicionário de equivalências."""
    chave = criar_chave_textual(valor)
    return mapa.get(chave, padrao)


def converter_moeda_brasileira(valor):
    """Converte números e textos como 'R$ 1.250,00' para float."""
    if pd.isna(valor) or str(valor).strip() == "":
        return np.nan

    if isinstance(valor, (int, float, np.integer, np.floating)):
        return float(valor)

    texto = str(valor).replace("R$", "").replace(" ", "").strip()

    # Quando há ponto e vírgula, assumimos padrão brasileiro: 1.250,00.
    if "." in texto and "," in texto:
        texto = texto.replace(".", "").replace(",", ".")
    elif "," in texto:
        texto = texto.replace(",", ".")

    return pd.to_numeric(texto, errors="coerce")


def email_e_valido(email):
    """Executa uma validação sintática simples; não confirma se o endereço existe."""
    if pd.isna(email):
        return False
    padrao = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"
    return bool(re.match(padrao, str(email).strip()))


def padronizar_canal(canal):
    """Padroniza os canais da API."""
    mapa = {
        "whatsapp": "WhatsApp",
        "e-mail": "E-mail",
        "email": "E-mail",
        "telefone": "Telefone",
        "portal": "Portal",
    }
    return mapa.get(criar_chave_textual(canal), "Outro")


print("Funções de transformação carregadas.")

## 6. Limpeza e padronização das matrículas

Decisões adotadas no exemplo:

- preservar colunas que indicam se houve imputação;
- não inventar e-mails ausentes ou inválidos;
- usar a mediana do curso para idade e mensalidade, somente para fins didáticos;
- manter o registro mais recente quando houver duplicidade de `aluno_id`;
- registrar datas impossíveis como ausentes, em vez de corrigi-las arbitrariamente.

In [ ]:
matriculas_limpas = matriculas_brutas.copy()

# Conversão das chaves e datas de atualização.
matriculas_limpas["aluno_id"] = pd.to_numeric(matriculas_limpas["aluno_id"], errors="coerce").astype("Int64")
matriculas_limpas["atualizado_em"] = pd.to_datetime(matriculas_limpas["atualizado_em"], errors="coerce")

# Padronização dos campos textuais.
matriculas_limpas["nome"] = matriculas_limpas["nome"].apply(padronizar_nome)
matriculas_limpas["cidade"] = matriculas_limpas["cidade"].apply(
    lambda valor: padronizar_com_mapa(valor, MAPA_CIDADES)
)
matriculas_limpas["curso"] = matriculas_limpas["curso"].apply(
    lambda valor: padronizar_com_mapa(valor, MAPA_CURSOS)
)
matriculas_limpas["status_matricula"] = matriculas_limpas["status_matricula"].apply(
    lambda valor: padronizar_com_mapa(valor, MAPA_STATUS)
)

# Conversão das datas. format='mixed' aceita os formatos variados do exemplo.
matriculas_limpas["data_matricula"] = pd.to_datetime(
    matriculas_limpas["data_matricula"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)

# Tratamento da idade.
matriculas_limpas["idade"] = pd.to_numeric(matriculas_limpas["idade"], errors="coerce")
matriculas_limpas["idade_originalmente_invalida"] = ~matriculas_limpas["idade"].between(18, 80)
matriculas_limpas.loc[~matriculas_limpas["idade"].between(18, 80), "idade"] = np.nan
matriculas_limpas["idade_foi_imputada"] = matriculas_limpas["idade"].isna()

# A mediana é calculada por curso para preservar diferenças entre grupos.
mediana_idade_por_curso = matriculas_limpas.groupby("curso")["idade"].transform("median")
matriculas_limpas["idade"] = matriculas_limpas["idade"].fillna(mediana_idade_por_curso)
matriculas_limpas["idade"] = matriculas_limpas["idade"].fillna(matriculas_limpas["idade"].median()).round().astype("Int64")

# Tratamento do valor monetário.
matriculas_limpas["valor_mensalidade"] = matriculas_limpas["valor_mensalidade"].apply(converter_moeda_brasileira)
matriculas_limpas["valor_foi_imputado"] = matriculas_limpas["valor_mensalidade"].isna()
mediana_valor_por_curso = matriculas_limpas.groupby("curso")["valor_mensalidade"].transform("median")
matriculas_limpas["valor_mensalidade"] = (
    matriculas_limpas["valor_mensalidade"]
    .fillna(mediana_valor_por_curso)
    .fillna(matriculas_limpas["valor_mensalidade"].median())
    .round(2)
)

# Limpeza e validação sintática do e-mail.
matriculas_limpas["email"] = matriculas_limpas["email"].apply(
    lambda valor: str(valor).strip().lower() if pd.notna(valor) else pd.NA
)
matriculas_limpas["email_valido"] = matriculas_limpas["email"].apply(email_e_valido)

# Primeiro removemos cópias exatamente iguais.
matriculas_limpas = matriculas_limpas.drop_duplicates()

# Depois mantemos a versão mais recente de cada aluno.
matriculas_limpas = (
    matriculas_limpas
    .sort_values(["aluno_id", "atualizado_em"])
    .drop_duplicates(subset=["aluno_id"], keep="last")
    .reset_index(drop=True)
)

print(f"Linhas na origem: {len(matriculas_brutas)}")
print(f"Alunos após deduplicação: {len(matriculas_limpas)}")
display(matriculas_limpas.head(8))

## 7. Tratamento das interações da API

A fonte JSON já foi achatada com `json_normalize`. Agora padronizamos tipos, categorias e datas antes da agregação por aluno.

In [ ]:
interacoes_limpas = interacoes_brutas.copy()

interacoes_limpas["interacao_id"] = pd.to_numeric(interacoes_limpas["interacao_id"], errors="coerce").astype("Int64")
interacoes_limpas["aluno_id"] = pd.to_numeric(interacoes_limpas["aluno_id"], errors="coerce").astype("Int64")
interacoes_limpas["data_hora"] = pd.to_datetime(interacoes_limpas["data_hora"], errors="coerce")
interacoes_limpas["detalhes_duracao_min"] = pd.to_numeric(
    interacoes_limpas["detalhes_duracao_min"], errors="coerce"
)
interacoes_limpas["canal"] = interacoes_limpas["canal"].apply(padronizar_canal)
interacoes_limpas["evento"] = interacoes_limpas["evento"].apply(padronizar_nome)

# Removemos duplicidades pela chave da interação.
interacoes_limpas = interacoes_limpas.drop_duplicates(subset=["interacao_id"], keep="last")

# Anti-join: identifica registros que não encontram correspondência na tabela principal.
ids_alunos_validos = set(matriculas_limpas["aluno_id"].dropna().astype(int))
interacoes_orfas = interacoes_limpas[
    ~interacoes_limpas["aluno_id"].isin(ids_alunos_validos)
].copy()

print(f"Interações válidas recebidas: {len(interacoes_limpas)}")
print(f"Interações órfãs encontradas: {len(interacoes_orfas)}")
display(interacoes_orfas)

# Agregamos as interações válidas para obter uma linha por aluno.
interacoes_agregadas = (
    interacoes_limpas[interacoes_limpas["aluno_id"].isin(ids_alunos_validos)]
    .groupby("aluno_id", as_index=False)
    .agg(
        total_interacoes=("interacao_id", "count"),
        ultima_interacao=("data_hora", "max"),
        duracao_total_min=("detalhes_duracao_min", "sum"),
        canais_utilizados=("canal", lambda valores: ", ".join(sorted(set(valores)))),
    )
)

display(interacoes_agregadas.head())

## 8. Extração de informação do texto

O texto não possui colunas prontas como um CSV. Usaremos uma expressão regular para localizar o identificador e uma regra simples de palavras-chave para classificar o assunto. Em projetos reais, essa etapa pode envolver técnicas mais avançadas de Processamento de Linguagem Natural.

In [ ]:
PADRAO_OBSERVACAO = re.compile(r"ALUNO_ID=(\d+)\s*;\s*(.+)")

TOPICOS = {
    "Bolsa": ["bolsa"],
    "Documentação": ["documento", "documentos"],
    "Financeiro": ["pagamento", "financeira", "financeiro", "negociação"],
    "Acessibilidade": ["acessibilidade"],
    "Portal e suporte": ["portal", "acesso", "suporte"],
    "Elogio": ["elogiou", "elogio"],
}


def identificar_topicos(texto):
    """Retorna os tópicos encontrados a partir de palavras-chave conhecidas."""
    chave = criar_chave_textual(texto) or ""
    encontrados = []

    for topico, palavras in TOPICOS.items():
        if any(criar_chave_textual(palavra) in chave for palavra in palavras):
            encontrados.append(topico)

    return ", ".join(encontrados) if encontrados else "Outros"


observacoes_extraidas = []

for linha in observacoes_brutas.splitlines():
    correspondencia = PADRAO_OBSERVACAO.match(linha.strip())

    if correspondencia:
        aluno_id = int(correspondencia.group(1))
        texto = correspondencia.group(2).strip()
        observacoes_extraidas.append(
            {
                "aluno_id": aluno_id,
                "observacao": texto,
                "topicos_observacao": identificar_topicos(texto),
            }
        )

observacoes_estruturadas = pd.DataFrame(observacoes_extraidas)
display(observacoes_estruturadas)

## 9. Integração das fontes

Utilizamos a matrícula como base principal e fazemos junções à esquerda (`left join`). Assim, nenhum aluno é eliminado apenas por não possuir interação ou observação.

In [ ]:
dados_integrados = (
    matriculas_limpas
    .merge(interacoes_agregadas, on="aluno_id", how="left", validate="one_to_one")
    .merge(observacoes_estruturadas, on="aluno_id", how="left", validate="one_to_one")
)

# Ausência de interação significa contagem zero; não significa dado desconhecido.
dados_integrados["total_interacoes"] = dados_integrados["total_interacoes"].fillna(0).astype(int)
dados_integrados["duracao_total_min"] = dados_integrados["duracao_total_min"].fillna(0).astype(int)
dados_integrados["canais_utilizados"] = dados_integrados["canais_utilizados"].fillna("Nenhum")
dados_integrados["topicos_observacao"] = dados_integrados["topicos_observacao"].fillna("Sem observação")

print(f"Quantidade de alunos após a integração: {len(dados_integrados)}")
display(
    dados_integrados[
        [
            "aluno_id", "nome", "curso", "status_matricula",
            "total_interacoes", "canais_utilizados", "topicos_observacao"
        ]
    ].head(10)
)

## 10. Validação do resultado

As verificações transformam expectativas em regras executáveis. Elas ajudam a impedir que dados inadequados avancem silenciosamente pelo pipeline.

In [ ]:
status_permitidos = {"Ativa", "Pendente", "Cancelada", "Não informado"}
cursos_permitidos = {
    "Ciência de Dados",
    "Engenharia de Software",
    "Inteligência Artificial",
    "Não informado",
}

verificacoes = pd.DataFrame(
    [
        {
            "verificacao": "aluno_id sem valores ausentes",
            "resultado": dados_integrados["aluno_id"].notna().all(),
            "detalhe": int(dados_integrados["aluno_id"].isna().sum()),
        },
        {
            "verificacao": "aluno_id único após deduplicação",
            "resultado": dados_integrados["aluno_id"].is_unique,
            "detalhe": int(dados_integrados["aluno_id"].duplicated().sum()),
        },
        {
            "verificacao": "idades entre 18 e 80 anos",
            "resultado": dados_integrados["idade"].between(18, 80).all(),
            "detalhe": int((~dados_integrados["idade"].between(18, 80)).sum()),
        },
        {
            "verificacao": "mensalidades não negativas",
            "resultado": dados_integrados["valor_mensalidade"].ge(0).all(),
            "detalhe": int((dados_integrados["valor_mensalidade"] < 0).sum()),
        },
        {
            "verificacao": "status pertencem ao domínio esperado",
            "resultado": set(dados_integrados["status_matricula"]).issubset(status_permitidos),
            "detalhe": sorted(set(dados_integrados["status_matricula"]) - status_permitidos),
        },
        {
            "verificacao": "cursos pertencem ao domínio esperado",
            "resultado": set(dados_integrados["curso"]).issubset(cursos_permitidos),
            "detalhe": sorted(set(dados_integrados["curso"]) - cursos_permitidos),
        },
    ]
)

verificacoes["status"] = np.where(verificacoes["resultado"], "APROVADO", "REPROVADO")
display(verificacoes[["verificacao", "status", "detalhe"]])

# Interrompe o pipeline se alguma regra crítica falhar.
assert verificacoes["resultado"].all(), "Uma ou mais validações críticas falharam."

print("Todas as validações críticas foram aprovadas.")

## 11. Armazenamento nas camadas Silver e Gold

- **Silver:** tabela integrada e limpa, ainda próxima do nível de detalhe operacional;
- **Gold:** dimensões, fatos e agregações preparadas para consumo analítico.

O SQLite foi escolhido por funcionar diretamente no Colab, sem servidor externo.

In [ ]:
# Salvamento da camada Silver.
caminho_silver = PASTA_SILVER / "alunos_integrados.csv"
dados_integrados.to_csv(caminho_silver, index=False, encoding="utf-8")

# ---------------------------
# Construção da camada Gold
# ---------------------------

# Dimensão de cursos: uma linha por curso.
dim_curso = pd.DataFrame({"curso": sorted(dados_integrados["curso"].unique())})
dim_curso.insert(0, "curso_sk", range(1, len(dim_curso) + 1))

# Dimensão de alunos: atributos descritivos do estudante.
dim_aluno = dados_integrados[
    ["aluno_id", "nome", "idade", "cidade", "email", "email_valido"]
].copy()
dim_aluno.insert(0, "aluno_sk", range(1, len(dim_aluno) + 1))

# Fato de matrícula: medidas e chaves para análise.
fato_matricula = (
    dados_integrados
    .merge(dim_aluno[["aluno_sk", "aluno_id"]], on="aluno_id", how="left", validate="one_to_one")
    .merge(dim_curso, on="curso", how="left", validate="many_to_one")
    [[
        "aluno_sk", "curso_sk", "data_matricula", "status_matricula",
        "valor_mensalidade", "total_interacoes", "duracao_total_min",
        "topicos_observacao"
    ]]
)

# Banco analítico local.
caminho_banco = PASTA_GOLD / "warehouse_educacional.db"

with sqlite3.connect(caminho_banco) as conexao:
    dim_aluno.to_sql("dim_aluno", conexao, if_exists="replace", index=False)
    dim_curso.to_sql("dim_curso", conexao, if_exists="replace", index=False)
    fato_matricula.to_sql("fato_matricula", conexao, if_exists="replace", index=False)

# Também exportamos as tabelas em CSV para facilitar a inspeção.
dim_aluno.to_csv(PASTA_GOLD / "dim_aluno.csv", index=False)
dim_curso.to_csv(PASTA_GOLD / "dim_curso.csv", index=False)
fato_matricula.to_csv(PASTA_GOLD / "fato_matricula.csv", index=False)

print(f"Arquivo Silver: {caminho_silver}")
print(f"Banco analítico: {caminho_banco}")
display(dim_curso)
display(fato_matricula.head())

## 12. Consumo por consulta SQL

A consulta reúne dimensão e fato para responder a uma pergunta de negócio: quantas matrículas existem e qual é o valor mensal total por curso e situação?

In [ ]:
consulta_sql = """
SELECT
    c.curso,
    f.status_matricula,
    COUNT(*) AS quantidade_matriculas,
    ROUND(SUM(f.valor_mensalidade), 2) AS valor_mensal_total,
    ROUND(AVG(f.total_interacoes), 2) AS media_interacoes
FROM fato_matricula AS f
INNER JOIN dim_curso AS c
    ON f.curso_sk = c.curso_sk
GROUP BY
    c.curso,
    f.status_matricula
ORDER BY
    c.curso,
    f.status_matricula;
"""

with sqlite3.connect(caminho_banco) as conexao:
    resultado_sql = pd.read_sql_query(consulta_sql, conexao)

display(resultado_sql)

## 13. Consumo por indicadores e visualizações

Os indicadores resumem a situação atual. Os gráficos permitem comparar categorias, mas não substituem a validação das definições e dos dados de origem.

In [ ]:
quantidade_alunos = dados_integrados["aluno_id"].nunique()
quantidade_ativas = int((dados_integrados["status_matricula"] == "Ativa").sum())
valor_ativo = dados_integrados.loc[
    dados_integrados["status_matricula"] == "Ativa", "valor_mensalidade"
].sum()
emails_pendentes = int((~dados_integrados["email_valido"]).sum())
percentual_com_interacao = (dados_integrados["total_interacoes"] > 0).mean() * 100

indicadores = pd.DataFrame(
    {
        "indicador": [
            "Alunos únicos",
            "Matrículas ativas",
            "Valor mensal das ativas",
            "E-mails ausentes ou inválidos",
            "Alunos com alguma interação",
        ],
        "valor": [
            quantidade_alunos,
            quantidade_ativas,
            f"R$ {valor_ativo:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."),
            emails_pendentes,
            f"{percentual_com_interacao:.1f}%",
        ],
    }
)

display(indicadores)

In [ ]:
# Estilo visual comum para os dois gráficos.
plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "font.size": 10,
    }
)

COR_AZUL = "#1F4E79"
COR_DOURADA = "#D9A441"
COR_CINZA = "#6B7280"

# Gráfico 1: quantidade de matrículas por status.
contagem_status = (
    dados_integrados["status_matricula"]
    .value_counts()
    .sort_values(ascending=True)
)

fig, eixo = plt.subplots()
contagem_status.plot(kind="barh", ax=eixo, color=COR_AZUL)
eixo.set_title("Quantidade de matrículas por status")
eixo.set_xlabel("Quantidade de alunos")
eixo.set_ylabel("Status da matrícula")
eixo.grid(axis="x", color="#E5E7EB", linewidth=0.8)
eixo.spines[["top", "right"]].set_visible(False)

for posicao, valor in enumerate(contagem_status.values):
    eixo.text(valor + 0.1, posicao, str(valor), va="center")

plt.tight_layout()
plt.show()

# Gráfico 2: valor mensal das matrículas ativas por curso.
valor_por_curso = (
    dados_integrados.query("status_matricula == 'Ativa'")
    .groupby("curso")["valor_mensalidade"]
    .sum()
    .sort_values(ascending=True)
)

fig, eixo = plt.subplots()
valor_por_curso.plot(kind="barh", ax=eixo, color=COR_DOURADA)
eixo.set_title("Valor mensal das matrículas ativas por curso")
eixo.set_xlabel("Valor mensal (R$)")
eixo.set_ylabel("Curso")
eixo.grid(axis="x", color="#E5E7EB", linewidth=0.8)
eixo.spines[["top", "right"]].set_visible(False)

for posicao, valor in enumerate(valor_por_curso.values):
    eixo.text(valor + 50, posicao, f"R$ {valor:,.0f}".replace(",", "."), va="center")

plt.tight_layout()
plt.show()

## 14. O pipeline construído

| Etapa | Implementação no notebook |
|---|---|
| Origem | CSV, JSON e TXT fictícios |
| Extração | `read_csv`, `json.load`, `json_normalize` e `read_text` |
| Qualidade | Nulos, duplicidades, domínios, tipos e registros órfãos |
| Transformação | Padronização, conversão, imputação controlada e deduplicação |
| Integração | Junções por `aluno_id` e agregações |
| Armazenamento | Camadas Bronze, Silver e Gold; CSV e SQLite |
| Consumo | SQL, indicadores e gráficos |

### Pontos importantes

- Engenharia de Dados não é apenas movimentar arquivos: envolve confiabilidade, rastreabilidade e regras de negócio.
- Dados ausentes não devem ser preenchidos sem justificativa.
- Uma junção pode eliminar ou multiplicar registros; por isso, a cardinalidade deve ser validada.
- A camada original deve ser preservada para permitir auditoria e reprocessamento.
- O consumidor final pode ser um relatório, um dashboard, uma aplicação ou um modelo de Inteligência Artificial.

# Atividade prática — Nova carga de matrículas

Uma nova unidade enviou o arquivo `novas_matriculas_atividade.csv`. O arquivo possui problemas de padronização, valores ausentes, duplicidade de chave e uma data inválida.

## Situação-problema

Você faz parte da equipe de Engenharia de Dados. Sua tarefa é preparar a nova carga para integração com a camada Silver, sem alterar os arquivos da camada Bronze.

## Entregas

1. Ler o arquivo mantendo os dados originais;
2. Apresentar o diagnóstico de qualidade;
3. Padronizar nome, cidade, curso, status, datas, idade, mensalidade e e-mail;
4. Tratar a duplicidade de `aluno_id`, mantendo o registro mais recente;
5. Identificar quais campos foram imputados ou permaneceram inválidos;
6. Concatenar a nova carga aos dados já tratados;
7. Garantir unicidade de `aluno_id` após a integração;
8. Salvar o resultado como `silver_com_nova_carga.csv`;
9. Produzir um indicador e um gráfico que ajudem a acompanhar a nova carga;
10. Responder às perguntas de reflexão ao final.

**Tempo sugerido:** 45 a 60 minutos.  
**Organização:** duplas ou trios.

In [ ]:
# Esta célula cria o arquivo que será entregue aos estudantes.
novas_matriculas_atividade = pd.DataFrame(
    [
        {
            "aluno_id": 1031,
            "nome": "  marta BEZERRA ",
            "idade": "29",
            "cidade": "RECIFE",
            "curso": "CD",
            "status_matricula": "ATIVO",
            "data_matricula": "19/09/2026",
            "valor_mensalidade": "R$ 1.250,00",
            "email": "MARTA.BEZERRA@EXEMPLO.COM ",
            "atualizado_em": "2026-09-19 08:00:00",
        },
        {
            "aluno_id": 1032,
            "nome": "Natan Soares",
            "idade": None,
            "cidade": "olinda",
            "curso": "inteligencia artificial",
            "status_matricula": "pendente",
            "data_matricula": "2026-09-20",
            "valor_mensalidade": 1380,
            "email": "natan.soares@exemplo.com",
            "atualizado_em": "2026-09-20 09:00:00",
        },
        {
            "aluno_id": 1033,
            "nome": "Olívia Ramos",
            "idade": "34",
            "cidade": "JABOATÃO",
            "curso": "eng. software",
            "status_matricula": "Cancelada",
            "data_matricula": "21-09-2026",
            "valor_mensalidade": "950,00",
            "email": "email_sem_formato",
            "atualizado_em": "2026-09-21 10:00:00",
        },
        {
            "aluno_id": 1034,
            "nome": "Pedro Xavier",
            "idade": "150",
            "cidade": None,
            "curso": "IA",
            "status_matricula": "Ativa",
            "data_matricula": "31/09/2026",
            "valor_mensalidade": None,
            "email": None,
            "atualizado_em": "2026-09-21 11:00:00",
        },
        {
            "aluno_id": 1032,
            "nome": "Natan Soares",
            "idade": "31",
            "cidade": "Olinda",
            "curso": "IA",
            "status_matricula": "Ativa",
            "data_matricula": "20/09/2026",
            "valor_mensalidade": "R$ 1.380,00",
            "email": "natan.soares@exemplo.com",
            "atualizado_em": "2026-09-22 14:00:00",
        },
    ]
)

caminho_atividade = PASTA_BRONZE / "novas_matriculas_atividade.csv"
novas_matriculas_atividade.to_csv(caminho_atividade, index=False, encoding="utf-8")

print(f"Arquivo da atividade criado em: {caminho_atividade}")
display(novas_matriculas_atividade)

## Espaço de desenvolvimento dos estudantes

Complete o roteiro abaixo. Reutilize as funções já construídas e explique, em comentários, as decisões tomadas.

In [ ]:
# ============================================================
# ATIVIDADE DOS ESTUDANTES — COMPLETE AS ETAPAS MARCADAS TODO
# ============================================================

# TODO 1: ler o arquivo caminho_atividade com dtype="object".
# nova_carga_bruta = ...

# TODO 2: executar diagnosticar_qualidade para a nova carga.
# diagnosticar_qualidade(...)

# TODO 3: criar uma cópia antes de transformar.
# nova_carga_limpa = ...

# TODO 4: converter aluno_id e atualizado_em.

# TODO 5: aplicar padronizar_nome, MAPA_CIDADES, MAPA_CURSOS e MAPA_STATUS.

# TODO 6: converter data_matricula com format="mixed" e errors="coerce".

# TODO 7: tratar idade inválida. Registre uma coluna booleana indicando imputação.

# TODO 8: converter e tratar valor_mensalidade. Registre a imputação.

# TODO 9: normalizar e validar o e-mail sem inventar novos endereços.

# TODO 10: ordenar por atualizado_em e remover duplicidades por aluno_id.

# TODO 11: integrar a nova carga com matriculas_limpas.
# Dica: concatene e deduplique novamente, pois uma carga futura poderia atualizar um aluno antigo.

# TODO 12: validar unicidade, idade, valores e domínios.

# TODO 13: salvar o resultado em PASTA_SILVER / "silver_com_nova_carga.csv".

# TODO 14: criar um indicador e um gráfico para apresentar à turma.

print("Roteiro disponibilizado. Nenhuma transformação foi executada nesta célula.")

## Perguntas de reflexão

1. Por que o primeiro registro do aluno `1032` não deve ser mantido como versão atual?
2. Por que não devemos inventar um e-mail para o aluno `1034`?
3. Qual é o risco de preencher todos os valores ausentes com zero?
4. O que deveria acontecer com a data impossível `31/09/2026`?
5. Como a equipe poderia impedir que as mesmas inconsistências voltassem a ocorrer na próxima carga?

## Critérios de avaliação — 10 pontos

| Critério | Pontuação |
|---|---:|
| Diagnóstico e identificação dos problemas | 2,0 |
| Padronização e conversão dos campos | 2,0 |
| Tratamento justificado de nulos e duplicidades | 2,0 |
| Integração e validações do resultado | 2,0 |
| Indicador, gráfico e clareza dos comentários | 2,0 |

## Gabarito do professor

> A seção seguinte deve ser removida ou ocultada antes de disponibilizar o notebook aos estudantes. Ela apresenta uma solução possível, não a única solução válida.

In [ ]:
# 1. Leitura e diagnóstico.
nova_carga_bruta = pd.read_csv(caminho_atividade, dtype="object")
diagnosticar_qualidade("Nova carga da atividade", nova_carga_bruta)

# 2. Cópia e conversão das chaves.
nova_carga_limpa = nova_carga_bruta.copy()
nova_carga_limpa["aluno_id"] = pd.to_numeric(nova_carga_limpa["aluno_id"], errors="coerce").astype("Int64")
nova_carga_limpa["atualizado_em"] = pd.to_datetime(nova_carga_limpa["atualizado_em"], errors="coerce")

# 3. Padronização de textos e categorias.
nova_carga_limpa["nome"] = nova_carga_limpa["nome"].apply(padronizar_nome)
nova_carga_limpa["cidade"] = nova_carga_limpa["cidade"].apply(
    lambda valor: padronizar_com_mapa(valor, MAPA_CIDADES)
)
nova_carga_limpa["curso"] = nova_carga_limpa["curso"].apply(
    lambda valor: padronizar_com_mapa(valor, MAPA_CURSOS)
)
nova_carga_limpa["status_matricula"] = nova_carga_limpa["status_matricula"].apply(
    lambda valor: padronizar_com_mapa(valor, MAPA_STATUS)
)
nova_carga_limpa["data_matricula"] = pd.to_datetime(
    nova_carga_limpa["data_matricula"], format="mixed", dayfirst=True, errors="coerce"
)

# 4. Idade: valores fora do domínio tornam-se ausentes e depois recebem a mediana da carga histórica.
nova_carga_limpa["idade"] = pd.to_numeric(nova_carga_limpa["idade"], errors="coerce")
nova_carga_limpa.loc[~nova_carga_limpa["idade"].between(18, 80), "idade"] = np.nan
nova_carga_limpa["idade_foi_imputada"] = nova_carga_limpa["idade"].isna()
mediana_idade_historica = float(matriculas_limpas["idade"].median())
nova_carga_limpa["idade"] = nova_carga_limpa["idade"].fillna(mediana_idade_historica).round().astype("Int64")
nova_carga_limpa["idade_originalmente_invalida"] = nova_carga_limpa["idade_foi_imputada"]

# 5. Mensalidade: conversão e imputação pela mediana histórica do respectivo curso.
nova_carga_limpa["valor_mensalidade"] = nova_carga_limpa["valor_mensalidade"].apply(converter_moeda_brasileira)
nova_carga_limpa["valor_foi_imputado"] = nova_carga_limpa["valor_mensalidade"].isna()
medianas_historicas_curso = matriculas_limpas.groupby("curso")["valor_mensalidade"].median()
nova_carga_limpa["valor_mensalidade"] = nova_carga_limpa.apply(
    lambda linha: (
        medianas_historicas_curso.get(linha["curso"], matriculas_limpas["valor_mensalidade"].median())
        if pd.isna(linha["valor_mensalidade"])
        else linha["valor_mensalidade"]
    ),
    axis=1,
)

# 6. E-mail: apenas limpeza e validação, sem criar valores fictícios.
nova_carga_limpa["email"] = nova_carga_limpa["email"].apply(
    lambda valor: str(valor).strip().lower() if pd.notna(valor) else pd.NA
)
nova_carga_limpa["email_valido"] = nova_carga_limpa["email"].apply(email_e_valido)

# 7. Deduplicação pela atualização mais recente.
nova_carga_limpa = (
    nova_carga_limpa
    .sort_values(["aluno_id", "atualizado_em"])
    .drop_duplicates(subset=["aluno_id"], keep="last")
    .reset_index(drop=True)
)

# 8. Integração com a camada histórica e nova deduplicação de segurança.
silver_com_nova_carga = (
    pd.concat([matriculas_limpas, nova_carga_limpa], ignore_index=True)
    .sort_values(["aluno_id", "atualizado_em"])
    .drop_duplicates(subset=["aluno_id"], keep="last")
    .reset_index(drop=True)
)

# 9. Validações essenciais.
assert silver_com_nova_carga["aluno_id"].is_unique
assert silver_com_nova_carga["idade"].between(18, 80).all()
assert silver_com_nova_carga["valor_mensalidade"].ge(0).all()
assert set(silver_com_nova_carga["status_matricula"]).issubset(status_permitidos)
assert set(silver_com_nova_carga["curso"]).issubset(cursos_permitidos)

# 10. Salvamento.
caminho_resultado_atividade = PASTA_SILVER / "silver_com_nova_carga.csv"
silver_com_nova_carga.to_csv(caminho_resultado_atividade, index=False, encoding="utf-8")

print(f"Nova carga limpa: {len(nova_carga_limpa)} alunos únicos.")
print(f"Silver atualizado: {len(silver_com_nova_carga)} alunos únicos.")
print(f"Resultado salvo em: {caminho_resultado_atividade}")
display(nova_carga_limpa)

In [ ]:
# Indicador e gráfico do gabarito.
resumo_nova_carga = (
    nova_carga_limpa["status_matricula"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="quantidade")
)

display(resumo_nova_carga)

fig, eixo = plt.subplots(figsize=(8, 4))
eixo.bar(resumo_nova_carga["status"], resumo_nova_carga["quantidade"], color=COR_AZUL)
eixo.set_title("Nova carga: quantidade de matrículas por status")
eixo.set_xlabel("Status")
eixo.set_ylabel("Quantidade")
eixo.set_ylim(0, max(resumo_nova_carga["quantidade"]) + 1)
eixo.grid(axis="y", color="#E5E7EB", linewidth=0.8)
eixo.spines[["top", "right"]].set_visible(False)

for indice, valor in enumerate(resumo_nova_carga["quantidade"]):
    eixo.text(indice, valor + 0.05, str(valor), ha="center")

plt.tight_layout()
plt.show()

## Encerramento

O notebook apresentou uma visão completa, mas introdutória, do trabalho de Engenharia de Dados. Nos próximos blocos, cada etapa será aprofundada:

- estratégias de extração completa e incremental;
- processamento em lote e em tempo real;
- regras avançadas de qualidade;
- ferramentas de ETL;
- carga em Data Warehouse;
- modelagem dimensional com tabelas fato e dimensão.

### Questão final para debate

**Qual etapa do pipeline causaria maior impacto no negócio se fosse executada incorretamente? Justifique considerando qualidade, custo, confiança e tomada de decisão.**